In [7]:
from IPython.display import HTML, display

live_emotion_tracker = """
<div style="text-align:center; font-family: 'Segoe UI', sans-serif;">
  <video id="webcam" autoplay playsinline muted style="display:none;"></video>
  <canvas id="output_canvas" style="border-radius:12px; max-width:100%; box-shadow: 0 4px 20px rgba(0,0,0,0.3);"></canvas>
  <h2 id="status" style="font-weight:500; color:#0d1b1e; margin-top:16px;">Loading models...</h2>
</div>

<script src="https://cdn.jsdelivr.net/npm/face-api.js@0.22.2/dist/face-api.min.js"></script>
<script>
(async function() {
  const video = document.getElementById("webcam");
  const canvas = document.getElementById("output_canvas");
  const status = document.getElementById("status");

  // Color per emotion: [border/pill color, soft fill color, dot color]
  const EMOTION_COLORS = {
    neutral:   { main: "#00C2CB", soft: "rgba(0, 194, 203, 0.15)",  dot: "#00E5D0" },
    happy:     { main: "#FFC107", soft: "rgba(255, 193, 7, 0.15)",  dot: "#FFD54F" },
    sad:       { main: "#3F51B5", soft: "rgba(63, 81, 181, 0.15)",  dot: "#5C6BC0" },
    angry:     { main: "#E53935", soft: "rgba(229, 57, 53, 0.15)",  dot: "#EF5350" },
    fearful:   { main: "#8E24AA", soft: "rgba(142, 36, 170, 0.15)", dot: "#AB47BC" },
    disgusted: { main: "#43A047", soft: "rgba(67, 160, 71, 0.15)",  dot: "#66BB6A" },
    surprised: { main: "#FB8C00", soft: "rgba(251, 140, 0, 0.15)",  dot: "#FFA726" }
  };

  const MODEL_URL = "https://cdn.jsdelivr.net/gh/justadudewhohacks/face-api.js@master/weights";

  status.textContent = "Loading face-api models...";
  await faceapi.nets.tinyFaceDetector.loadFromUri(MODEL_URL);
  await faceapi.nets.faceExpressionNet.loadFromUri(MODEL_URL);
  await faceapi.nets.faceLandmark68Net.loadFromUri(MODEL_URL);

  status.textContent = "Requesting camera access...";
  const stream = await navigator.mediaDevices.getUserMedia({ video: true });
  video.srcObject = stream;
  await video.play();

  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;

  const displaySize = { width: canvas.width, height: canvas.height };
  faceapi.matchDimensions(canvas, displaySize);

  status.textContent = "Live";
  const ctx = canvas.getContext("2d");

  function drawRoundedRect(x, y, w, h, r) {
    ctx.beginPath();
    ctx.moveTo(x + r, y);
    ctx.arcTo(x + w, y, x + w, y + h, r);
    ctx.arcTo(x + w, y + h, x, y + h, r);
    ctx.arcTo(x, y + h, x, y, r);
    ctx.arcTo(x, y, x + w, y, r);
    ctx.closePath();
  }

  async function renderLoop() {
    const detections = await faceapi
      .detectAllFaces(video, new faceapi.TinyFaceDetectorOptions())
      .withFaceLandmarks()
      .withFaceExpressions();

    const resized = faceapi.resizeResults(detections, displaySize);

    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.drawImage(video, 0, 0, canvas.width, canvas.height);

    let statusText = "No face detected";

    resized.forEach(det => {
      const box = det.detection.box;
      const expressions = det.expressions;
      const sorted = Object.entries(expressions).sort((a, b) => b[1] - a[1]);
      const [topEmotion, topScore] = sorted[0];
      const label = `${topEmotion.charAt(0).toUpperCase() + topEmotion.slice(1)} · ${(topScore * 100).toFixed(0)}%`;

      const colors = EMOTION_COLORS[topEmotion] || EMOTION_COLORS.neutral;

      // Soft-filled bounding box with thin colored border
      ctx.fillStyle = colors.soft;
      ctx.strokeStyle = colors.main;
      ctx.lineWidth = 2;
      drawRoundedRect(box.x, box.y, box.width, box.height, 14);
      ctx.fill();
      ctx.stroke();

      // Minimal landmark dots, matching tone
      ctx.fillStyle = colors.dot;
      det.landmarks.positions.forEach(pt => {
        ctx.beginPath();
        ctx.arc(pt.x, pt.y, 1.6, 0, 2 * Math.PI);
        ctx.fill();
      });

      // Pill-style label above the box
      ctx.font = "600 16px 'Segoe UI', sans-serif";
      const textWidth = ctx.measureText(label).width;
      const pillPadding = 12;
      const pillWidth = textWidth + pillPadding * 2;
      const pillHeight = 30;
      const pillX = box.x;
      const pillY = box.y - pillHeight - 8;

      ctx.fillStyle = colors.main;
      drawRoundedRect(pillX, pillY, pillWidth, pillHeight, pillHeight / 2);
      ctx.fill();

      ctx.fillStyle = "#062226";
      ctx.textBaseline = "middle";
      ctx.fillText(label, pillX + pillPadding, pillY + pillHeight / 2 + 1);

      statusText = label;
    });

    status.textContent = statusText;
    requestAnimationFrame(renderLoop);
  }

  renderLoop();
})();
</script>
"""

display(HTML(live_emotion_tracker))